In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy
import numpy as np
import io
from PIL import Image
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas

In [ ]:
df = pd.read_csv("/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/GPR37L1/GPR37L1-LacZ_VGLUT1-PSD95_output_data/results.csv")
df.columns

In [ ]:
# adding a brain column to the dataframe
df['Brain'] = df['img_filename'].str.split('_').str[2]
df.head(2)

In [ ]:
class PlotResults:

    def __init__(self, df, candidate_gRNA, name_of_plot, control_gRNA = "LacZ-gRNA", ):
        self.df = df
        self.candidate_gRNA = candidate_gRNA    
        self.control_gRNA = "LacZ-gRNA"
        self.name_of_plot = name_of_plot

        """
        Args:
            df: the dataframe with all the data
            candidate_gRNA: which is the candidate_gRNA from which you want to check the statistics
            control_gRNA: is standard LacZ-gRNA
            name_of_plot: name of the plot provided
        """

    def check_statistics(self, hippocampal_layer, metric):
        """
        This function checks the different between hemispheres per brain. Uses a paired t-test to compare the means.

        Args:
            df: the dataframe with all the data
            hippocampal_layer: which hippocampal layer to check the statistics from
            metric: whichc metric to check the statistics from
            candidate_gRNA: which is the candidate_gRNA from which you want to check the statistics
            control_gRNA: is standard LacZ-gRNA
        
        Returns:
            filtered_df_hip_layer: the filtered dataframe with the hippocampal layer
            replicate_averages_long: dataframe with the means calculated from each brain, in compact format
            replicate_averages_wide: dataframe with the means calculated from each brain, in long format
            statistic: the T statistic of the paired t-test
            pvalue: the accompanying p-value of the paired t-test
        """
        filtered_df_hip_layer = self.df[self.df["hippocampal_layer"].str.contains(hippocampal_layer)]
        replicate_averages_long = filtered_df_hip_layer.groupby(["gRNA", "Brain"], as_index = False).agg({metric:"mean"})
        replicate_averages_wide = replicate_averages_long.pivot_table(columns = "gRNA", values = metric, index = "Brain")
        statistic, pvalue = scipy.stats.ttest_rel(replicate_averages_wide[self.control_gRNA], replicate_averages_wide[self.candidate_gRNA])
        
        return filtered_df_hip_layer, replicate_averages_long, replicate_averages_wide, statistic, pvalue
    

    def create_scatter_plot_with_means(self, ax, df_averages, df_points, metric):
        """"
        Creates a single scatter plot (one metric & one hippocampal layer), where the brain means are plotted and connected with lines.
        Points on the side represent each individual measured data point.
        
        Args:
            ax: no need to fullfill argument, is used for plotting the mean and individual points.
            df_averages: dataframe with the means calculated from each brain, in long format (relates to the "replicate_averages_long" output from check_statistics).
            df_points: dataframe that contains the individual points per brain from a specific hippocampal layer (related to the "filtered_df_hip_layer" output from check statistics).
            metric: which metric to plot.
            candidate_gRNA: which is the candidate_gRNA from which you want to check the statistics.
            control_gRNA: is standard LacZ-gRNA.

        Returns:
            A single scatter plot where one metric is plotted for one hippocampal layer. 
        """
        # individual points - preparing the data for plotting
        df_points_LacZ_gRNA = df_points[df_points["gRNA"] == self.control_gRNA]
        df_points_cand_gRNA = df_points[df_points["gRNA"] == self.candidate_gRNA]

        # mean points - preparing the data for plotting
        df_average_LacZ_gRNA = df_averages[df_averages["gRNA"] == self.control_gRNA]
        df_average_cand_gRNA = df_averages[df_averages["gRNA"] == self.candidate_gRNA]

        # set the individual points across the x axis
        x_LacZ_points = np.ones(len(df_points_LacZ_gRNA)) * 0.1  
        x_VCAM1_points = np.ones(len(df_points_cand_gRNA)) * 0.9  

        # set the mean points across the x axis
        x_LacZ_averages = np.ones(len(df_average_LacZ_gRNA)) * 0.25  
        x_VCAM1_averages = np.ones(len(df_average_cand_gRNA)) * 0.75 

        # plot the individual points
        ax.scatter(x_LacZ_points, df_points_LacZ_gRNA[metric], color = "#808080", edgecolors = "black", alpha = 0.6, linewidths = 0.5)
        ax.scatter(x_VCAM1_points, df_points_cand_gRNA[metric], color = "#c92ffb", edgecolors = "black", alpha = 0.6, linewidths = 0.5)

        # plot the mean points
        ax.scatter(x_LacZ_averages, df_average_LacZ_gRNA[metric], color = "#808080", edgecolors = "black", linewidths = 0.8)
        ax.scatter(x_VCAM1_averages, df_average_cand_gRNA[metric], color = "#c92ffb", edgecolors = "black", linewidths = 0.8)

        # plot the lines connecting the mean points
        for i in range(len(df_average_LacZ_gRNA)):
            ax.plot([0.25, 0.75], [df_average_LacZ_gRNA.iloc[i][metric], df_average_cand_gRNA.iloc[i][metric]], linewidth = 0.5, c = "k")

        # other attributes
        ax.spines[['right', 'top']].set_visible(False)


    def create_scatter_plot_with_means_per_hippocampal_layer(self, hippocampal_layer_list, metric, ax = None):
        """
        Creates a scatter plot (one metric & all hippocampal layers), where the brain means are plotted and connected with lines. 
        Points on the side represent each individual measured data point.
        Also plots the p value above the each comparison and a star if significant. 
        
        Args:
            df: the large dataframe containing all the metric measurement from all the hippocampal layers, needs to have a column that specifies which brains is measured.
            hippocampal_layer_list: list of hippocampal layers that needs to be plotted.
            metric: which metric to plot.
            candidate_gRNA: which is the candidate_gRNA from which you want to check the statistics.
            control_gRNA: is standard LacZ-gRNA.
        
        Returns:
            Scatter plots where one metric is plotted for all hippocampal layers. 
        """

        # set the size of the figure
        if ax is None:
            fig, axes = plt.subplots(nrows = 1, ncols = len(hippocampal_layer_list), figsize=(12, 6))

        # variables to track the global y-axis limits
        global_y_max = float('-inf')

        for idx, hippocampal_layer in enumerate(hippocampal_layer_list):
            
            # get statistics
            df_filtered, df_averages, _, _, p_value = self.check_statistics(hippocampal_layer = hippocampal_layer, metric = metric)


            # update global y-axis limits
            local_y_max = max(df_filtered[metric].max(), df_averages[metric].max())
            global_y_max = max(global_y_max, local_y_max)
            
            # make the plot on the specific subplot axis
            ax

            # make the plot on the specific subplot axis
            ax = axes[idx]
            self.create_scatter_plot_with_means(ax, df_averages = df_averages, df_points = df_filtered, metric = metric)

            # set x-axis labels to hippocampal layer names
            ax.set_xlabel(f"{hippocampal_layer}")

            # remove x-tick labels
            ax.set_xticks([])

            # remove y-axis labels and y axis lines for all but the first plot
            if idx > 0:
                ax.set_ylabel("")
                ax.set_yticks([])
                ax.spines['left'].set_visible(False)

            # add p-value to each graph
            p_value_str = f"p = {p_value:.3f}"
            p_value_text = ax.text(0.5, 0.95, p_value_str, ha = 'center', va='bottom', transform=ax.transAxes, fontsize=12)  # Font size for p-value

            # add significant stars as a separate text object with a larger font size
            stars = ""
            if p_value < 0.0001:
                stars = "****"
            elif p_value < 0.001:
                stars = "***"
            elif p_value < 0.01:
                stars = "**"
            elif p_value < 0.05:
                stars = "*"
            stars_text = ax.text(0.5, 0.98, stars, ha='center', va='bottom', transform=ax.transAxes, fontsize=20)  # Larger font size for stars

        # apply global y-axis limits to all subplots
        global_y_max = global_y_max * 1.1  # adjust the maximum y-axis to be 0.1 higher
        for ax in axes:
            ax.set_ylim(0, global_y_max)

        # create a color-coded legend outside the loop with patch objects for color matching
        handles = [
            plt.Rectangle((0, 0), 1, 1, color="#808080", ec="black", lw=0.5),
            plt.Rectangle((0, 0), 1, 1, color="#c92ffb", ec="black", lw=0.5),
        ]
        labels = [self.control_gRNA, self.candidate_gRNA]
        fig.legend(handles, labels, loc='upper left', title="gRNA", bbox_to_anchor=(1, 1))  # Adjust x and y coordinates as needed

        # Add a title to the figure
        fig.suptitle(metric, fontsize=18)

        plt.tight_layout()

        return fig

    def save_plot_to_bytes(self, fig):
        """
        Helper function that saves an image into buffer memory. 
        This is way around the axes from matplotlib because its already used to create the figure from the
        "create_scatter_plot_with_means_per_hippocampal_layer" function.

        Args:
            fig: scatter plots where one metric is plotted for all hippocampal layers (relates to the output from "create_scatter_plot_with_means_per_hippocampal_layer" )
        
        Returns:
            image of the object.
        """
        buf = io.BytesIO()
        canvas = FigureCanvas(fig)
        fig.savefig(buf, format='png', bbox_inches='tight') # to ensure that the legend for each plot is printed out.
        buf.seek(0)
        return Image.open(buf)
    

    def save_figure(self, hippocampal_layer_list, metric_list):
        """
        This is the master function, where it all comes together to save the plot.

        Args:
            hippocampal_layer_list: list of hippocampal layers assessed
            metric_list: list of metrics assessed.
        """
        # loops over the images
        images = []
        for metric in metric_list:
            fig = self.create_scatter_plot_with_means_per_hippocampal_layer(hippocampal_layer_list = hippocampal_layer_list, metric = metric)
            image = self.save_plot_to_bytes(fig)
            images.append(image)
            plt.close(fig) # do not print out each figure

        # extracts figure dimensions
        widths, heights = zip(*(img.size for img in images))
        total_height = sum(heights)
        max_width = max(widths) +1

        # combines the figures and pastes it into the blank canvas
        combined_image = Image.new('RGB', (max_width, total_height))
        y_offset = 0
        for img in images:
            combined_image.paste(img, (0, y_offset))
            y_offset += img.height

        # saves the image
        combined_image.save(self.name_of_plot + ".png")

In [ ]:
merged_df =df

In [ ]:


import sys
sys.path.append("/Users/cgeyskens/Documents/code/phd/image-analysis/synapse-counting")
from synapse_counting import results_plotting

In [ ]:
# ### ---------------------- pre wrangling for statistics and plotting per brain -------------------------------------- ###
merged_df['Brain'] = merged_df['img_filename'].str.split('_').str[2] # create new column with brain replicates

hippocampal_layers = ["CA1 SLM", "CA1 SR", "CA1 SO", "CA3 SO", "CA3 SL", "CA3 SR", "DG ML", "DG Hilus"]
metrics = ["overlap_um2", "pearson_cor", "overlap_coeff", "presynapse_image_mfi", "postsynapse_image_mfi", "pre_puncta_density_per_100_um2", "post_puncta_density_per_100_um2", "pre_staining_area_um2", "post_staining_area_um2", "pre_mean_puncta_size_um2", "post_mean_puncta_size_um2"]


# ### -------------------------------------------- calculate statistics ----------------------------------------------- ###

# calculate the statistics with a t-test
statistcs_instance = results_plotting.PlotResults(df = merged_df, candidate_gRNA = "GPR37L1-gRNA", name_of_plot = "doesnt_matter")
statistics_results = []
for hippocampal_layer in hippocampal_layers:
    for metric in metrics:
        _ , _, table , _ , p_value = statistcs_instance.check_statistics(hippocampal_layer = hippocampal_layer, metric = metric)
        statistics_results.append({"hippocampal_layer": hippocampal_layer, "metric": metric, "p_value": p_value})


In [ ]:
### This part is for testing for eventually to the package files

In [ ]:
def check_statistics(df, hippocampal_layer, metric, candidate_gRNA, control_gRNA = "LacZ-gRNA"):
    """
    This function checks the different between hemispheres per brain. Uses a paired t-test to compare the means.

    Args:
        df: the dataframe with all the data
        hippocampal_layer: which hippocampal layer to check the statistics from
        metric: whichc metric to check the statistics from
        candidate_gRNA: which is the candidate_gRNA from which you want to check the statistics
        control_gRNA: is standard LacZ-gRNA
    
    Returns:
        filtered_df_hip_layer: the filtered dataframe with the hippocampal layer
        replicate_averages_long: dataframe with the means calculated from each brain, in compact format
        replicate_averages_wide: dataframe with the means calculated from each brain, in long format
        statistic: the T statistic of the paired t-test
        pvalue: the accompanying p-value of the paired t-test
    """
    filtered_df_hip_layer = df[df["hippocampal_layer"].str.contains(hippocampal_layer)]
    replicate_averages_long = filtered_df_hip_layer.groupby(["gRNA", "Brain"], as_index = False).agg({metric:"mean"})
    replicate_averages_wide = replicate_averages_long.pivot_table(columns = "gRNA", values = metric, index = "Brain")
    statistic, pvalue = scipy.stats.ttest_rel(replicate_averages_wide[control_gRNA], replicate_averages_wide[candidate_gRNA])
    
    return filtered_df_hip_layer, replicate_averages_long, replicate_averages_wide, statistic, pvalue

In [ ]:
a, b, c , d, e = check_statistics(df, hippocampal_layer = "CA3 SL", metric = "presynapse_image_mfi", candidate_gRNA = "VCAM1-gRNA")
c

In [ ]:
def create_scatter_plot_with_means(ax, df_averages, df_points, metric, candidate_gRNA, control_gRNA = "LacZ-gRNA"):
    """"
    Creates a single scatter plot (one metric & one hippocampal layer), where the brain means are plotted and connected with lines.
    Points on the side represent each individual measured data point.
    
    Args:
        ax: no need to fullfill argument, is used for plotting the mean and individual points.
        df_averages: dataframe with the means calculated from each brain, in long format (relates to the "replicate_averages_long" output from check_statistics).
        df_points: dataframe that contains the individual points per brain from a specific hippocampal layer (related to the "filtered_df_hip_layer" output from check statistics).
        metric: which metric to plot.
        candidate_gRNA: which is the candidate_gRNA from which you want to check the statistics.
        control_gRNA: is standard LacZ-gRNA.

    Returns:
        A single scatter plot where one metric is plotted for one hippocampal layer. 
    """
    # individual points - preparing the data for plotting
    df_points_LacZ_gRNA = df_points[df_points["gRNA"] == control_gRNA]
    df_points_cand_gRNA = df_points[df_points["gRNA"] == candidate_gRNA]

    # mean points - preparing the data for plotting
    df_average_LacZ_gRNA = df_averages[df_averages["gRNA"] == control_gRNA]
    df_average_cand_gRNA = df_averages[df_averages["gRNA"] == candidate_gRNA]

    # set the individual points across the x axis
    x_LacZ_points = np.ones(len(df_points_LacZ_gRNA)) * 0.1  
    x_VCAM1_points = np.ones(len(df_points_cand_gRNA)) * 0.9  

    # set the mean points across the x axis
    x_LacZ_averages = np.ones(len(df_average_LacZ_gRNA)) * 0.25  
    x_VCAM1_averages = np.ones(len(df_average_cand_gRNA)) * 0.75 

    # plot the individual points
    ax.scatter(x_LacZ_points, df_points_LacZ_gRNA[metric], color = "#808080", edgecolors = "black", alpha = 0.6, linewidths = 0.5)
    ax.scatter(x_VCAM1_points, df_points_cand_gRNA[metric], color = "#c92ffb", edgecolors = "black", alpha = 0.6, linewidths = 0.5)

    # plot the mean points
    ax.scatter(x_LacZ_averages, df_average_LacZ_gRNA[metric], color = "#808080", edgecolors = "black", linewidths = 0.8)
    ax.scatter(x_VCAM1_averages, df_average_cand_gRNA[metric], color = "#c92ffb", edgecolors = "black", linewidths = 0.8)

    # plot the lines connecting the mean points
    for i in range(len(df_average_LacZ_gRNA)):
        ax.plot([0.25, 0.75], [df_average_LacZ_gRNA.iloc[i][metric], df_average_cand_gRNA.iloc[i][metric]], linewidth = 0.5, c = "k")

    # other attributes
    ax.spines[['right', 'top']].set_visible(False)

In [ ]:
def create_scatter_plot_with_means_per_hippocampal_layer(df, hippocampal_layer_list, metric, candidate_gRNA, control_gRNA = "LacZ-gRNA", ax = None):
    """
    Creates a scatter plot (one metric & all hippocampal layers), where the brain means are plotted and connected with lines. 
    Points on the side represent each individual measured data point.
    Also plots the p value above the each comparison and a star if significant. 
    
    Args:
        df: the large dataframe containing all the metric measurement from all the hippocampal layers, needs to have a column that specifies which brains is measured.
        hippocampal_layer_list: list of hippocampal layers that needs to be plotted.
        metric: which metric to plot.
        candidate_gRNA: which is the candidate_gRNA from which you want to check the statistics.
        control_gRNA: is standard LacZ-gRNA.
    
    Returns:
        Scatter plots where one metric is plotted for all hippocampal layers. 
    """

    # set the size of the figure
    if ax is None:
        fig, axes = plt.subplots(nrows = 1, ncols = len(hippocampal_layer_list), figsize=(12, 6))

    # variables to track the global y-axis limits
    global_y_max = float('-inf')

    for idx, hippocampal_layer in enumerate(hippocampal_layer_list):
        
        # get statistics
        df_filtered, df_averages, _, _, p_value = check_statistics(df = df, hippocampal_layer = hippocampal_layer, metric = metric, candidate_gRNA = candidate_gRNA)
        
        print("Columns in df_filtered:", df_filtered.columns)
        print("Columns in df_averages:", df_averages.columns)


        # update global y-axis limits
        local_y_max = max(df_filtered[metric].max(), df_averages[metric].max())
        global_y_max = max(global_y_max, local_y_max)
        
        # make the plot on the specific subplot axis
        ax

        # make the plot on the specific subplot axis
        ax = axes[idx]
        create_scatter_plot_with_means(ax, df_averages = df_averages, df_points = df_filtered, metric = metric, candidate_gRNA = candidate_gRNA)

        # set x-axis labels to hippocampal layer names
        ax.set_xlabel(f"{hippocampal_layer}")

        # remove x-tick labels
        ax.set_xticks([])

        # remove y-axis labels and y axis lines for all but the first plot
        if idx > 0:
            ax.set_ylabel("")
            ax.set_yticks([])
            ax.spines['left'].set_visible(False)

        # add p-value to each graph
        p_value_str = f"p = {p_value:.3f}"
        p_value_text = ax.text(0.5, 0.95, p_value_str, ha = 'center', va='bottom', transform=ax.transAxes, fontsize=12)  # Font size for p-value

        # add significant stars as a separate text object with a larger font size
        stars = ""
        if p_value < 0.0001:
            stars = "****"
        elif p_value < 0.001:
            stars = "***"
        elif p_value < 0.01:
            stars = "**"
        elif p_value < 0.05:
            stars = "*"
        stars_text = ax.text(0.5, 0.98, stars, ha='center', va='bottom', transform=ax.transAxes, fontsize=20)  # Larger font size for stars

    # apply global y-axis limits to all subplots
    global_y_max = global_y_max * 1.1  # adjust the maximum y-axis to be 0.1 higher
    for ax in axes:
        ax.set_ylim(0, global_y_max)

    # create a color-coded legend outside the loop with patch objects for color matching
    handles = [
        plt.Rectangle((0, 0), 1, 1, color="#808080", ec="black", lw=0.5),
        plt.Rectangle((0, 0), 1, 1, color="#c92ffb", ec="black", lw=0.5),
    ]
    labels = [control_gRNA, candidate_gRNA]
    fig.legend(handles, labels, loc='upper left', title="gRNA", bbox_to_anchor=(1, 1))  # Adjust x and y coordinates as needed

    # Add a title to the figure
    fig.suptitle(metric, fontsize=18)

    plt.tight_layout()

    return fig

In [ ]:
def save_plot_to_bytes(fig):
    """
    Saves an image into buffer memory. 
    This is way around the axes from matplotlib because its already used to create the figure from the
    "create_scatter_plot_with_means_per_hippocampal_layer" function.

    Args:
        fig: scatter plots where one metric is plotted for all hippocampal layers (relates to the output from "create_scatter_plot_with_means_per_hippocampal_layer" )
    
    Returns:
        image of the object
    """
    buf = io.BytesIO()
    canvas = FigureCanvas(fig)
    fig.savefig(buf, format='png', bbox_inches='tight')  # Ensure the entire figure including legend is saved
    buf.seek(0)
    return Image.open(buf)

In [ ]:
hippocampal_layers = ["CA1 SLM", "CA1 SR", "CA1 SO", "CA3 SO", "CA3 SL", "CA3 SR", "DG ML", "DG Hilus"]
metrics = ["overlap_um2", "pearson_cor", "overlap_coeff", "presynapse_image_mfi", "postsynapse_image_mfi", "pre_puncta_density_per_100_um2", "post_puncta_density_per_100_um2", "pre_staining_area_um2", "post_staining_area_um2", "pre_mean_puncta_size_um2", "post_mean_puncta_size_um2"]

#Call the function for each metric
plot_presynapse_image_mfi = create_scatter_plot_with_means_per_hippocampal_layer(df = df, hippocampal_layer_list = hippocampal_layers, metric = "presynapse_image_mfi", candidate_gRNA = "VCAM1-gRNA")

In [ ]:
hippocampal_layers = ["CA1 SLM", "CA1 SR", "CA1 SO", "CA3 SO", "CA3 SL", "CA3 SR", "DG ML", "DG Hilus"]
metrics = ["overlap_um2", "pearson_cor", "overlap_coeff", "presynapse_image_mfi", "postsynapse_image_mfi", "pre_puncta_density_per_100_um2", "post_puncta_density_per_100_um2", "pre_staining_area_um2", "post_staining_area_um2", "pre_mean_puncta_size_um2", "post_mean_puncta_size_um2"]

# loops over the images
images = []
for metric in metrics:
    fig = create_scatter_plot_with_means_per_hippocampal_layer(df = df, hippocampal_layer_list = hippocampal_layers, metric = metric, candidate_gRNA="VCAM1-gRNA")
    image = save_plot_to_bytes(fig)
    images.append(image)
    # plt.close(fig) # do not print out each figure

# extracts figure dimensions
widths, heights = zip(*(img.size for img in images))
total_height = sum(heights)
max_width = max(widths)

# combines the figures and pastes it into the blank canvas
combined_image = Image.new('RGB', (max_width, total_height))
y_offset = 0
for img in images:
    combined_image.paste(img, (0, y_offset))
    y_offset += img.height

# saves the image
combined_image.save("combined_plot.png")

In [ ]:
def save_figure(df, hippocampal_layer_list, metric_list, candidate_gRNA, control_gRNA = "LacZ-gRNA"):
    # loops over the images
    images = []
    for metric in metric_list:
        fig = create_scatter_plot_with_means_per_hippocampal_layer(df = df, hippocampal_layer_list = hippocampal_layer_list, metric = metric, candidate_gRNA = candidate_gRNA, control_gRNA = control_gRNA)
        image = save_plot_to_bytes(fig)
        images.append(image)
        plt.close(fig) # do not print out each figure

    # extracts figure dimensions
    widths, heights = zip(*(img.size for img in images))
    total_height = sum(heights)
    max_width = max(widths) +1

    # combines the figures and pastes it into the blank canvas
    combined_image = Image.new('RGB', (max_width, total_height))
    y_offset = 0
    for img in images:
        combined_image.paste(img, (0, y_offset))
        y_offset += img.height

    # saves the image
    combined_image.save("combined_plot.png")

In [ ]:
hippocampal_layers = ["CA1 SLM", "CA1 SR", "CA1 SO", "CA3 SO", "CA3 SL", "CA3 SR", "DG ML", "DG Hilus"]
metrics = ["overlap_um2", "pearson_cor", "overlap_coeff", "presynapse_image_mfi", "postsynapse_image_mfi", "pre_puncta_density_per_100_um2", "post_puncta_density_per_100_um2", "pre_staining_area_um2", "post_staining_area_um2", "pre_mean_puncta_size_um2", "post_mean_puncta_size_um2"]

save_figure(df = df, hippocampal_layer_list = hippocampal_layers, metric_list = metrics, candidate_gRNA = "VCAM1-gRNA", control_gRNA = "LacZ-gRNA")

In [ ]:
##############################

In [ ]:
def check_statistics(df, hippocampal_layer, metric, candidate_gRNA, control_gRNA = "LacZ-gRNA"):
    filtered_df_hip_layer = df[df["hippocampal_layer"].str.contains(hippocampal_layer)]
    ReplicateAverages = filtered_df_hip_layer.groupby(["gRNA","Brain"], as_index=False).agg({metric:"mean"})
    ReplicateAveragesPivot = ReplicateAverages.pivot_table(columns="gRNA", values=metric, index="Brain")
    statistic, pvalue = scipy.stats.ttest_rel(ReplicateAveragesPivot[control_gRNA], ReplicateAveragesPivot[candidate_gRNA])
    
    return filtered_df_hip_layer, ReplicateAveragesPivot, ReplicateAverages, statistic, pvalue

In [ ]:
_ , ReplicateAveragesPivot, ReplicateAverages , _, p_value = check_statistics(df, hippocampal_layer = "CA3 SL", metric = "presynapse_image_mfi", candidate_gRNA = "VCAM1-gRNA")
ReplicateAverages

In [ ]:
hippocampal_layers  =  ["CA1 SLM", "CA1 SR", "CA1 SO", "CA3 SO", "CA3 SL", "CA3 SR", "DG ML", "DG Hilus"]
metrics = ["overlap_um2", "pearson_cor", "overlap_coeff", "presynapse_image_mfi", "postsynapse_image_mfi", "pre_puncta_density_per_100_um2", "post_puncta_density_per_100_um2", "pre_staining_area_um2", "post_staining_area_um2", "pre_mean_puncta_size_um2", "post_mean_puncta_size_um2"]

statistics_results = []

for hippocampal_layer in hippocampal_layers:
    for metric in metrics:
        _ , table, _ , _ , p_value = check_statistics(df, hippocampal_layer = hippocampal_layer, metric = metric, candidate_gRNA = "VCAM1-gRNA")
        statistics_results.append({"hippocampal_layer": hippocampal_layer, "metric": metric, "p_value": p_value})

df_statistics_results = pd.DataFrame(statistics_results)  # create dataFrame from list
print(df_statistics_results)

In [ ]:
p_value_threshold = 0.05  # Set your desired p-value threshold
filtered_df = df_statistics_results[df_statistics_results["p_value"] <= p_value_threshold]

# Print the filtered DataFrame
print(filtered_df)

In [ ]:
df_filtered , table, df_averages , _ , p_value = check_statistics(df, hippocampal_layer = "CA3 SL", metric = "presynapse_image_mfi", candidate_gRNA = "VCAM1-gRNA")
df_filtered

In [ ]:
df_cand_gRNA = df_filtered[df_filtered["gRNA"] == "VCAM1-gRNA"]
df_LacZ_gRNA = df_filtered[df_filtered["gRNA"] == "LacZ-gRNA"]

df_average_cand_gRNA = df_averages[df_averages["gRNA"] == "VCAM1-gRNA"]
df_average_LacZ_gRNA = df_averages[df_averages["gRNA"] == "LacZ-gRNA"]

fig, ax = plt.subplots(figsize=(2, 6))

x_LacZ_points = np.ones(len(df_LacZ_gRNA)) * 0.1  
x_VCAM1_points = np.ones(len(df_cand_gRNA)) * 0.9  

x_LacZ = np.ones(len(df_average_LacZ_gRNA)) * 0.2  
x_VCAM1 = np.ones(len(df_average_cand_gRNA)) * 0.8  

plt.scatter(x_LacZ_points, df_LacZ_gRNA["presynapse_image_mfi"], color = "#808080", label="LacZ-gRNA", edgecolors= "black", alpha = 0.8, linewidths = 0.5)
plt.scatter(x_VCAM1_points, df_cand_gRNA["presynapse_image_mfi"], color = "#c92ffb", label="VCAM1-gRNA", edgecolors= "black", alpha = 0.8, linewidths = 0.5)

plt.scatter(x_LacZ, df_average_LacZ_gRNA["presynapse_image_mfi"], color = "#808080", label="LacZ-gRNA", edgecolors= "black", linewidths = 0.8)
plt.scatter(x_VCAM1, df_average_cand_gRNA["presynapse_image_mfi"], color = "#c92ffb", label="VCAM1-gRNA", edgecolors= "black", linewidths = 0.8)

for i in range(len(df_average_LacZ_gRNA)):
    plt.plot([0.2, 0.8], [df_average_LacZ_gRNA.iloc[i]["presynapse_image_mfi"], df_average_cand_gRNA.iloc[i]["presynapse_image_mfi"]], linewidth = 0.5, c='k')

plt.xticks(ticks = [0, 0.2, 0.8, 1], 
           labels = ["", 'LacZ-gRNA', "VCAM1-gRNA", ""], 
           minor = False)
plt.title("CA3 SL: presynapse_image_mfi")

plt.ylabel("presynapse_image_mfi")
plt.xlabel("gRNA")
ax.spines[['right', 'top']].set_visible(False)
plt.ylim(bottom = 0)

plt.show()

In [ ]:
def create_scatter_plot_with_means(df_averages, df_points, metric, title, candidate_gRNA, control_gRNA = "LacZ-gRNA"):
    
    # individual points - preparing the data for plotting
    df_points_LacZ_gRNA = df_points[df_points["gRNA"] == control_gRNA]
    df_points_cand_gRNA = df_points[df_points["gRNA"] == candidate_gRNA]

    # mean points - preparing the data for plotting
    df_average_LacZ_gRNA = df_averages[df_averages["gRNA"] == control_gRNA]
    df_average_cand_gRNA = df_averages[df_averages["gRNA"] == candidate_gRNA]

    # set the size of the figure
    fig, ax = plt.subplots(figsize=(2, 6))
    
    # set the individual points across the x axis
    x_LacZ_points = np.ones(len(df_points_LacZ_gRNA)) * 0.1  
    x_VCAM1_points = np.ones(len(df_points_cand_gRNA)) * 0.9  

    # set the mean points across the x axis
    x_LacZ_averages = np.ones(len(df_average_LacZ_gRNA)) * 0.2  
    x_VCAM1_averages = np.ones(len(df_average_cand_gRNA)) * 0.8  

    # plot the individual points
    ax.scatter(x_LacZ_points, df_points_LacZ_gRNA[metric], color = "#808080", label = control_gRNA, edgecolors= "black", alpha = 0.6, linewidths = 0.5)
    ax.scatter(x_VCAM1_points, df_points_cand_gRNA[metric], color = "#c92ffb", label = candidate_gRNA, edgecolors= "black", alpha = 0.6, linewidths = 0.5)

    # plot the mean points
    ax.scatter(x_LacZ_averages, df_average_LacZ_gRNA[metric], color = "#808080", label = control_gRNA, edgecolors= "black", linewidths = 0.8)
    ax.scatter(x_VCAM1_averages, df_average_cand_gRNA[metric], color = "#c92ffb", label= candidate_gRNA, edgecolors= "black", linewidths = 0.8)

    # plot the lines connecting the mean points
    for i in range(len(df_average_LacZ_gRNA)):
        plt.plot([0.2, 0.8], [df_average_LacZ_gRNA.iloc[i][metric], df_average_cand_gRNA.iloc[i][metric]], linewidth = 0.5, c = "k")

    # plot the ticks
    ax.set_xticks(ticks = [0, 0.2, 0.8, 1])
    ax.set_xticklabels(["", control_gRNA, candidate_gRNA, ""])

    # other attributes
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.set_xlabel("gRNA")
    ax.spines[['right', 'top']].set_visible(False)
    ax.set_ylim(bottom = 0)

    return fig


In [ ]:
df_filtered , _ , df_averages , _ , p_value = check_statistics(df, hippocampal_layer = "CA3 SL", metric = "postsynapse_image_mfi", candidate_gRNA = "VCAM1-gRNA")

p = create_scatter_plot_with_means(df_averages = df_averages, df_points = df_filtered, metric = "postsynapse_image_mfi", title = "CA3 SL: postsynapse_image_mfi", candidate_gRNA = "VCAM1-gRNA")

In [ ]:

hippocampal_layers = ["CA1 SLM", "CA1 SR", "CA1 SO", "CA3 SO", "CA3 SL", "CA3 SR", "DG ML", "DG Hilus"]
fig, axes = plt.subplots(nrows=1, ncols= len(hippocampal_layers), figsize=(20, 5))

for idx, hippocampal_layer in enumerate(hippocampal_layers):
    # get the statistics
    df_filtered , _ , df_averages , _ , p_value = check_statistics(df, hippocampal_layer = hippocampal_layer, metric = "presynapse_image_mfi" , candidate_gRNA = "VCAM1-gRNA")
    # make the plot
    ax = axes[idx] 
    create_scatter_plot_with_means(df_averages = df_averages, df_points = df_filtered, metric = "presynapse_image_mfi", title = f"{hippocampal_layer}: presynapse_image_mfi", candidate_gRNA = "VCAM1-gRNA")
    
plt.tight_layout()



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Assuming check_statistics and create_scatter_plot_with_means are defined elsewhere
# from your_module import check_statistics

# Define your DataFrame `df` here or ensure it's imported correctly
# df = pd.read_csv("your_data.csv")  # Example of loading data

hippocampal_layers = ["CA1 SLM", "CA1 SR", "CA1 SO", "CA3 SO", "CA3 SL", "CA3 SR", "DG ML", "DG Hilus"]

# set the size of the figure
fig, axes = plt.subplots(nrows = 1, ncols = len(hippocampal_layers), figsize=(14, 6))

def create_scatter_plot_with_means(ax, df_averages, df_points, metric, candidate_gRNA, control_gRNA = "LacZ-gRNA"):
    
    # individual points - preparing the data for plotting
    df_points_LacZ_gRNA = df_points[df_points["gRNA"] == control_gRNA]
    df_points_cand_gRNA = df_points[df_points["gRNA"] == candidate_gRNA]

    # mean points - preparing the data for plotting
    df_average_LacZ_gRNA = df_averages[df_averages["gRNA"] == control_gRNA]
    df_average_cand_gRNA = df_averages[df_averages["gRNA"] == candidate_gRNA]

    # set the individual points across the x axis
    x_LacZ_points = np.ones(len(df_points_LacZ_gRNA)) * 0.1  
    x_VCAM1_points = np.ones(len(df_points_cand_gRNA)) * 0.9  

    # set the mean points across the x axis
    x_LacZ_averages = np.ones(len(df_average_LacZ_gRNA)) * 0.25  
    x_VCAM1_averages = np.ones(len(df_average_cand_gRNA)) * 0.75 

    # plot the individual points
    ax.scatter(x_LacZ_points, df_points_LacZ_gRNA[metric], color = "#808080", edgecolors = "black", alpha = 0.6, linewidths = 0.5)
    ax.scatter(x_VCAM1_points, df_points_cand_gRNA[metric], color = "#c92ffb", edgecolors = "black", alpha = 0.6, linewidths = 0.5)

    # plot the mean points
    ax.scatter(x_LacZ_averages, df_average_LacZ_gRNA[metric], color = "#808080", edgecolors = "black", linewidths = 0.8)
    ax.scatter(x_VCAM1_averages, df_average_cand_gRNA[metric], color = "#c92ffb", edgecolors = "black", linewidths = 0.8)

    # plot the lines connecting the mean points
    for i in range(len(df_average_LacZ_gRNA)):
        ax.plot([0.25, 0.75], [df_average_LacZ_gRNA.iloc[i][metric], df_average_cand_gRNA.iloc[i][metric]], linewidth = 0.5, c = "k")

    # other attributes
    ax.spines[['right', 'top']].set_visible(False)

# variables to track the global y-axis limits
global_y_max = float('-inf')

for idx, hippocampal_layer in enumerate(hippocampal_layers):
    # get statistics
    df_filtered, _, df_averages, _, p_value = check_statistics(df, hippocampal_layer=hippocampal_layer, metric="presynapse_image_mfi", candidate_gRNA="VCAM1-gRNA")
    
    # update global y-axis limits
    local_y_max = max(df_filtered["presynapse_image_mfi"].max(), df_averages["presynapse_image_mfi"].max())
    global_y_max = max(global_y_max, local_y_max)
    
    # make the plot on the specific subplot axis
    ax

    # make the plot on the specific subplot axis
    ax = axes[idx]
    create_scatter_plot_with_means(ax, df_averages, df_filtered, metric="presynapse_image_mfi", candidate_gRNA="VCAM1-gRNA")

    # set x-axis labels to hippocampal layer names
    ax.set_xlabel(f"{hippocampal_layer}")

    # remove x-tick labels
    ax.set_xticks([])

    # remove y-axis labels and y axis lines for all but the first plot
    if idx > 0:
        ax.set_ylabel("")
        ax.set_yticks([])
        ax.spines['left'].set_visible(False)

    # add p-value to each graph
    p_value_str = f"p = {p_value:.3f}"
    p_value_text = ax.text(0.5, 0.95, p_value_str, ha = 'center', va='bottom', transform=ax.transAxes, fontsize=12)  # Font size for p-value

    # add significant stars as a separate text object with a larger font size
    stars = ""
    if p_value < 0.0001:
        stars = "****"
    elif p_value < 0.001:
        stars = "***"
    elif p_value < 0.01:
        stars = "**"
    elif p_value < 0.05:
        stars = "*"
    stars_text = ax.text(0.5, 0.98, stars, ha='center', va='bottom', transform=ax.transAxes, fontsize=20)  # Larger font size for stars

# apply global y-axis limits to all subplots
global_y_max = global_y_max * 1.1  # adjust the maximum y-axis to be 0.1 higher
for ax in axes:
    ax.set_ylim(0, global_y_max)

# create a color-coded legend outside the loop with patch objects for color matching
handles = [
    plt.Rectangle((0, 0), 1, 1, color="#808080", ec="black", lw=0.5),
    plt.Rectangle((0, 0), 1, 1, color="#c92ffb", ec="black", lw=0.5),
]
labels = ["LacZ-gRNA", "VCAM1-gRNA"]
fig.legend(handles, labels, loc='upper left', title="gRNA", bbox_to_anchor=(1, 1))  # Adjust x and y coordinates as needed

# Add a title to the figure
fig.suptitle("presynapse_image_mfi", fontsize=18)

plt.tight_layout()
plt.show()  # Or plt.savefig("output.png") to save the figure

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import io
from PIL import Image
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas

def create_scatter_plot_with_means(ax, df_averages, df_points, metric, candidate_gRNA, control_gRNA = "LacZ-gRNA"):
    
    # individual points - preparing the data for plotting
    df_points_LacZ_gRNA = df_points[df_points["gRNA"] == control_gRNA]
    df_points_cand_gRNA = df_points[df_points["gRNA"] == candidate_gRNA]

    # mean points - preparing the data for plotting
    df_average_LacZ_gRNA = df_averages[df_averages["gRNA"] == control_gRNA]
    df_average_cand_gRNA = df_averages[df_averages["gRNA"] == candidate_gRNA]

    # set the individual points across the x axis
    x_LacZ_points = np.ones(len(df_points_LacZ_gRNA)) * 0.1  
    x_VCAM1_points = np.ones(len(df_points_cand_gRNA)) * 0.9  

    # set the mean points across the x axis
    x_LacZ_averages = np.ones(len(df_average_LacZ_gRNA)) * 0.25  
    x_VCAM1_averages = np.ones(len(df_average_cand_gRNA)) * 0.75 

    # plot the individual points
    ax.scatter(x_LacZ_points, df_points_LacZ_gRNA[metric], color = "#808080", edgecolors = "black", alpha = 0.6, linewidths = 0.5)
    ax.scatter(x_VCAM1_points, df_points_cand_gRNA[metric], color = "#c92ffb", edgecolors = "black", alpha = 0.6, linewidths = 0.5)

    # plot the mean points
    ax.scatter(x_LacZ_averages, df_average_LacZ_gRNA[metric], color = "#808080", edgecolors = "black", linewidths = 0.8)
    ax.scatter(x_VCAM1_averages, df_average_cand_gRNA[metric], color = "#c92ffb", edgecolors = "black", linewidths = 0.8)

    # plot the lines connecting the mean points
    for i in range(len(df_average_LacZ_gRNA)):
        ax.plot([0.25, 0.75], [df_average_LacZ_gRNA.iloc[i][metric], df_average_cand_gRNA.iloc[i][metric]], linewidth = 0.5, c = "k")

    # other attributes
    ax.spines[['right', 'top']].set_visible(False)



def create_scatter_plot_with_means_per_hippocampal_layer(df, hippocampal_layer_list, metric, candidate_gRNA, control_gRNA = "LacZ-gRNA", ax = None):
    
    # set the size of the figure
    if ax is None:
        fig, axes = plt.subplots(nrows = 1, ncols = len(hippocampal_layer_list), figsize=(12, 6))

    # variables to track the global y-axis limits
    global_y_max = float('-inf')

    for idx, hippocampal_layer in enumerate(hippocampal_layer_list):
        
        # get statistics
        df_filtered, _, df_averages, _, p_value = check_statistics(df=df, hippocampal_layer = hippocampal_layer, metric= metric, candidate_gRNA = candidate_gRNA)
        
        # update global y-axis limits
        local_y_max = max(df_filtered[metric].max(), df_averages[metric].max())
        global_y_max = max(global_y_max, local_y_max)
        
        # make the plot on the specific subplot axis
        ax

        # make the plot on the specific subplot axis
        ax = axes[idx]
        create_scatter_plot_with_means(ax, df_averages = df_averages, df_points = df_filtered, metric = metric, candidate_gRNA = candidate_gRNA)

        # set x-axis labels to hippocampal layer names
        ax.set_xlabel(f"{hippocampal_layer}")

        # remove x-tick labels
        ax.set_xticks([])

        # remove y-axis labels and y axis lines for all but the first plot
        if idx > 0:
            ax.set_ylabel("")
            ax.set_yticks([])
            ax.spines['left'].set_visible(False)

        # add p-value to each graph
        p_value_str = f"p = {p_value:.3f}"
        p_value_text = ax.text(0.5, 0.95, p_value_str, ha = 'center', va='bottom', transform=ax.transAxes, fontsize=12)  # Font size for p-value

        # add significant stars as a separate text object with a larger font size
        stars = ""
        if p_value < 0.0001:
            stars = "****"
        elif p_value < 0.001:
            stars = "***"
        elif p_value < 0.01:
            stars = "**"
        elif p_value < 0.05:
            stars = "*"
        stars_text = ax.text(0.5, 0.98, stars, ha='center', va='bottom', transform=ax.transAxes, fontsize=20)  # Larger font size for stars

    # apply global y-axis limits to all subplots
    global_y_max = global_y_max * 1.1  # adjust the maximum y-axis to be 0.1 higher
    for ax in axes:
        ax.set_ylim(0, global_y_max)

    # create a color-coded legend outside the loop with patch objects for color matching
    handles = [
        plt.Rectangle((0, 0), 1, 1, color="#808080", ec="black", lw=0.5),
        plt.Rectangle((0, 0), 1, 1, color="#c92ffb", ec="black", lw=0.5),
    ]
    labels = [control_gRNA, candidate_gRNA]
    fig.legend(handles, labels, loc='upper left', title="gRNA", bbox_to_anchor=(1, 1))  # Adjust x and y coordinates as needed

    # Add a title to the figure
    fig.suptitle(metric, fontsize=18)

    plt.tight_layout()

    return fig

# saves an image into buffer memory, way around the axes from matplotlib because its already used to create a single
def save_plot_to_bytes(fig):
    buf = io.BytesIO()
    canvas = FigureCanvas(fig)
    canvas.print_png(buf)
    buf.seek(0)
    return Image.open(buf)

In [ ]:
hippocampal_layers = ["CA1 SLM", "CA1 SR", "CA1 SO", "CA3 SO", "CA3 SL", "CA3 SR", "DG ML", "DG Hilus"]
metrics = ["overlap_um2", "pearson_cor", "overlap_coeff", "presynapse_image_mfi", "postsynapse_image_mfi", "pre_puncta_density_per_100_um2", "post_puncta_density_per_100_um2", "pre_staining_area_um2", "post_staining_area_um2", "pre_mean_puncta_size_um2", "post_mean_puncta_size_um2"]


# Call the function for each metric
plot_presynapse_image_mfi = create_scatter_plot_with_means_per_hippocampal_layer(df = df, hippocampal_layer_list = hippocampal_layers, metric = "presynapse_image_mfi", candidate_gRNA = "VCAM1-gRNA")
plot_postsynapse_image_mfi = create_scatter_plot_with_means_per_hippocampal_layer(df = df, hippocampal_layer_list = hippocampal_layers, metric = "postsynapse_image_mfi", candidate_gRNA = "VCAM1-gRNA")


In [ ]:
hippocampal_layers = ["CA1 SLM", "CA1 SR", "CA1 SO", "CA3 SO", "CA3 SL", "CA3 SR", "DG ML", "DG Hilus"]
metrics = ["overlap_um2", "pearson_cor", "overlap_coeff", "presynapse_image_mfi", "postsynapse_image_mfi", "pre_puncta_density_per_100_um2", "post_puncta_density_per_100_um2", "pre_staining_area_um2", "post_staining_area_um2", "pre_mean_puncta_size_um2", "post_mean_puncta_size_um2"]

# loops over the images
images = []
for metric in metrics:
    fig = create_scatter_plot_with_means_per_hippocampal_layer(hippocampal_layers, df_averages, df_filtered, metric, candidate_gRNA="VCAM1-gRNA")
    image = save_plot_to_bytes(fig)
    images.append(image)
    plt.close(fig) # do not print out each figure

# extracts figure dimensions
widths, heights = zip(*(img.size for img in images))
total_height = sum(heights)
max_width = max(widths)

# combines the figures and pastes it into the blank canvas
combined_image = Image.new('RGB', (max_width, total_height))
y_offset = 0
for img in images:
    combined_image.paste(img, (0, y_offset))
    y_offset += img.height

# saves the image
combined_image.save("combined_plot.png")


In [ ]:
fig, ax = plt.subplots(figsize=(3, 6))

sns.swarmplot(x="gRNA", y="overlap_um2", hue="Brain", alpha=0.9, size= 7, palette="deep", data=df_filtered)

# line as mean
mean = table.mean()
for i, m in enumerate(mean):
    plt.hlines(m, i - 0.3, i + 0.3, color='black', linewidth=1)

plt.text(0.3, 0.2, "p-value")
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.title("CA3 SO: overlap_um2")
plt.ylim(bottom = 0)

In [ ]:
def plot_significant_data(df_filtered, metric, title, p_value):
    
    fig, ax = plt.subplots(figsize=(3, 6))
    p = sns.swarmplot(x="gRNA", y=metric, hue="Brain", alpha=0.9, size= 7, palette="deep", data=df_filtered)
    
    mean = table.mean()
    for i, m in enumerate(mean):
        plt.hlines(m, i - 0.3, i + 0.3, color='black', linewidth=1)
    p_value = round(p_value, 4)
    p.text(0.2, 0.0715, f"p-val = {p_value}")
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    p.set_title(title)
    p.set_ylim(bottom = 0)
    
    return p

In [ ]:
plot_significant_data(df_filtered = df_filtered, metric = "post_mean_puncta_size_um2", p_value = p_value, title = "CA3 SO: postsynapse puncta size")

In [ ]:
filtered_df_CA3_SL = df[df['hippocampal_layer'].str.contains('CA3 SL')]

In [ ]:
# statistics and getting into the right format - CA3 SL
ReplicateAverages = filtered_df_CA3_SL.groupby(["gRNA","Brain"], as_index=False).agg({"presynapse_image_mfi":"mean"})
ReplicateAverages.head(5)
ReplicateAvePivot = ReplicateAverages.pivot_table(columns="gRNA", values="presynapse_image_mfi", index="Brain")
ReplicateAvePivot.head(10)

In [ ]:
statistic, pvalue = scipy.stats.ttest_rel(ReplicateAvePivot["LacZ-gRNA"], ReplicateAvePivot["VCAM1-gRNA"])
print("pvalue:", pvalue, "\n" "& t-statistic:", statistic)
mean = ReplicateAvePivot.mean()
print(mean)
sem = ReplicateAvePivot.sem()
print(sem)